<a href="https://colab.research.google.com/github/gmauricio-toledo/tda-gdl/blob/main/02-Exploracion_alta_dimensionalidad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exploración en espacios de alta dimensión

## Puntos aleatorios en el espacio

Puntos con una distribución uniforme

In [ ]:
import numpy as np

a, b = -1.0, 1.0

puntos = np.random.uniform(low=a, high=b, size=(1000, 300))
print(puntos.shape)
print(puntos[:5,:3])

Puntos con una distribución normal

In [ ]:
import numpy as np

media = 0
desviacion = 1

puntos = np.random.normal(loc=media,
                          scale=desviacion,
                          size=(1000, 300))
print(puntos[:5,:3])

## Visualización

¿Qué función estamos usando para reducir la dimensionalidad?

$$f:\mathbb{R}^D\rightarrow \mathbb{R}^2$$
$$f(x)=...$$

In [ ]:
import matplotlib.pyplot as plt

i, j = 10, 46 # Escogamos dos componentes para graficar

plt.figure()
plt.scatter(puntos[:,i], puntos[:,j])
plt.show()

# Datasets reales

## Regresión

Un dataset para un problema de regresión.

Este conjunto de datos contiene información sobre distintos distritos de California y fue elaborado a partir del censo de Estados Unidos de 1990, utilizando como unidad geográfica los grupos de manzanas censales, que son las áreas más pequeñas para las cuales el Censo proporciona datos muestrales. Cada grupo típicamente tiene entre 600 y 3.000 habitantes. El dataset incluye 20.640 observaciones con 8 atributos numéricos y una variable objetivo. Los atributos describen características como el ingreso mediano por grupo de manzanas, la edad mediana de las viviendas, el número promedio de habitaciones y dormitorios por hogar, la población total, el número promedio de personas por hogar, y las coordenadas geográficas (latitud y longitud). La variable objetivo es el valor mediano de las viviendas en cada distrito, expresado en cientos de miles de dólares. Este conjunto de datos no tiene valores faltantes y es ampliamente utilizado para tareas de regresión en aprendizaje automático.

In [ ]:
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()
print(housing.DESCR)

In [ ]:
type(housing)

Extraemos las variables, tanto las independientes (features o características) como la dependiente (target).

In [ ]:
housing_data = housing.data
housing_prices = housing.target
print(housing_data.shape)
print(housing_prices.shape)

In [ ]:
housing_data[:3,:5]

In [ ]:
housing_prices[:3]

### Visualización

In [ ]:
import matplotlib.pyplot as plt

i, j = 2, 3  # Escojamos dos variables

plt.figure()
plt.scatter(housing_data[:,i], housing_data[:,j], c=housing_prices, cmap='viridis')
plt.colorbar()
plt.show()

## Clasificación

Este es un conjunto de datos de 60,000 imágenes en escala de grises de 28x28 de los 10 dígitos, junto con un conjunto de prueba de 10,000 imágenes.

In [ ]:
from keras.datasets import mnist

(x_train, y_train), (x_test, y_test) = mnist.load_data()

print(x_train.shape)
print(y_train.shape)
print(x_test.shape)
print(y_test.shape)

In [ ]:
y_train[:5]

Hacemos un reshape

In [ ]:
x_train = x_train.reshape(x_train.shape[0], -1)
x_test = x_test.reshape(x_test.shape[0], -1)
print(x_train.shape)
print(x_test.shape)

In [ ]:
import matplotlib.pyplot as plt

idxs = np.random.choice(x_train.shape[0], size=5, replace=False) # ¿qué estamos haciendo aquí?

plt.figure(figsize=(8, 4))
for i, idx in enumerate(idxs):
    plt.subplot(1, 5, i+1)
    plt.imshow(x_train[idx].reshape(28, 28), cmap='gray')
    plt.title(f'Label: {y_train[idx]}')
    plt.axis('off')
plt.show()

In [ ]:
np.unique(y_train)

In [ ]:
np.random.choice(x_train.shape[0], size=2, replace=False)

In [ ]:
import matplotlib.pyplot as plt

two_idxs = np.random.choice(x_train.shape[1], size=2, replace=False)
i = two_idxs[0]
j = two_idxs[1]

plt.figure()
for label in np.unique(y_train):
    xs = x_train[y_train == label,i]
    ys = x_train[y_train == label,j]
    plt.scatter(xs, ys, label=label)
plt.suptitle(f'Variables {i} y {j}')
plt.legend(loc='best')
plt.show()

¿Cuál es la función que hace la reducción de dimensionalidad aquí?

In [ ]:
D = x_train.shape[1]
d = 2

A = np.random.uniform(low=-5.0, high=5.0, size=(d, D))
A_digits = np.dot(A,x_train.transpose()).transpose()
print(A_digits.shape)


In [ ]:
A_digits[:5,:]

In [ ]:
plt.figure()
for label in np.unique(y_train):
    xs = A_digits[y_train == label,0]
    ys = A_digits[y_train == label,1]
    plt.scatter(xs, ys, label=label)
plt.legend(loc='best')
plt.show()

# El algoritmo `vecinos más cercanos`

Ahora exploraremos los vecinos más cercanos de un punto en un espacio de dimensión alta y verificaremos si su cercanía en el espacio de características corresponde a cercanía en el fenómeno que describe.

Para esto usaremos la clase `NearestNeighbors` de scikit learn. Observemos su inicialización, los hiperparámetros por defecto (por ejemplo, la métrica) y el uso de la clase entrenada.

In [ ]:
from keras.datasets import mnist

(x_train, y_train), (x_test, y_test) = mnist.load_data()

x_train = x_train.reshape(x_train.shape[0], -1)
x_test = x_test.reshape(x_test.shape[0], -1)

In [ ]:
from sklearn.neighbors import NearestNeighbors

nn = NearestNeighbors(n_neighbors=5)
nn.fit(x_train)

In [ ]:
import matplotlib.pyplot as plt

test_idx = 431

plt.figure(figsize=(4,4))
plt.imshow(x_test[test_idx].reshape(28, 28), cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
x_test[test_idx].reshape(1,-1)

Veamos quién es el más cercano

In [ ]:
nn.kneighbors(x_test[test_idx].reshape(1,-1))

In [ ]:
distances, idxs = nn.kneighbors(x_test[test_idx].reshape(1,-1))

for d, idx in zip(distances[0], idxs[0]):
    print(f'Distancia: {d:.2f} - Etiqueta: {y_train[idx]}')
    plt.figure()
    plt.imshow(x_train[idx].reshape(28, 28), cmap='gray')
    plt.axis('off')
    plt.show()

## El pipeline del aprendizaje automático

El clasificador `KNeighborsClassifier` implementa el algoritmo descrito en el paso anterior a todo el dataset de prueba y evalua su desempeño en la tarea.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

clf = KNeighborsClassifier(n_neighbors=5)
clf.fit(x_train, y_train)

Observa los efectos de la dimensionalidad en el desempeño del clasificador

In [ ]:
clf.score(x_test, y_test)

Veamos la predicción de un elemento en particular

In [ ]:
clf.predict(x_test[43].reshape(1,-1))

In [ ]:
plt.figure(figsize=(4,4))
plt.imshow(x_test[43].reshape(28, 28), cmap='gray')
plt.axis('off')
plt.show()